# Conjugate gradients for the Poisson equation

Discretising the Poisson equation on an `m` by `m` grid gives a linear system `A x = b`. The
matrix `A` is never stored. It acts on a grid of values by multiplying each value by 4 and
subtracting its four neighbours, where a neighbour beyond the edge of the grid counts as zero.
The conjugate gradient method improves a guess for `x` step by step. Each step needs one
application of `A`, two dot products and three vector updates, and each step depends on the one
before it. The function below starts from `x = 0` and takes a fixed number of steps, so that the
answer is the same on every run. The demonstration solves a system whose right-hand side is 1
everywhere, and prints the size of the residual `b - A x` after 60 steps.

In [ ]:
import numpy as np


def poisson_operator(v):
    """Apply the five-point operator to a grid, counting values beyond the edge as zero."""
    out = 4.0 * v
    out[1:, :] -= v[:-1, :]
    out[:-1, :] -= v[1:, :]
    out[:, 1:] -= v[:, :-1]
    out[:, :-1] -= v[:, 1:]
    return out


def cgsolve(b, iters):
    """Take `iters` conjugate gradient steps towards the solution of A x = b, from x = 0."""
    x = np.zeros_like(b)
    r = b.copy()  # the residual b - A x
    p = r.copy()  # the search direction
    rr = np.vdot(r, r)
    for _ in range(iters):
        q = poisson_operator(p)
        alpha = rr / np.vdot(p, q)
        x += alpha * p
        r -= alpha * q
        rr_new = np.vdot(r, r)
        p = r + (rr_new / rr) * p
        rr = rr_new
    return x

In [ ]:
print(np.linalg.norm(np.ones((32, 32)) - poisson_operator(cgsolve(np.ones((32, 32)), 60))))

The declaration below tells the harness how to call the function and how to check a routine
against it. The result `x` is not an argument, so the C routine receives an array in which to
write it, and the C routine takes `x`, `b`, the side `m` of the grid and the number of steps, in
that order. `make_inputs` draws a right-hand side of random numbers between 0 and 1, a few of
them exactly 0 or 1, and never all zero. It takes between 50 and 100 steps, but never more steps
than there are unknowns. In exact arithmetic the method finds the exact answer after as many
steps as there are unknowns, and a further step could then divide zero by zero.

The tolerance is 1e-11 of the largest element of the solution. This is about twelve times the
largest difference, 8e-13, between the function and a routine that adds up the dot products in a
different order, on a grid of side 1024.

In [ ]:
def make_inputs(size, rng):
    b = rng.uniform(0.0, 1.0, (size, size))
    special = rng.random((size, size)) < 0.05
    b[special] = rng.integers(0, 2, special.sum())
    b[0, 0] = rng.uniform(0.5, 1.0)  # the right-hand side must not be all zero
    unknowns = size * size
    return {"b": b, "iters": int(rng.integers(min(unknowns, 50), min(unknowns, 100) + 1))}


ROUTINE = {
    "function": cgsolve,
    "make_inputs": make_inputs,
    "arguments": {"b": "float64[m, m]", "iters": "int32"},
    "results": {"x": "float64[m, m]"},
    "c_order": ["x", "b", "m", "iters"],
    "test_sizes": [1, 8, 64, 256],
    "benchmark_size": 1024,
    "tolerance": 1e-11,
}